In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 19:26:50 WARN Utils: Your hostname, Ethans-Laptop.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.4 instead (on interface en0)
26/03/05 19:26:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 19:26:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_green = spark.read.option("recursiveFileLookup","true").parquet(
"/Users/ethandu/data-engineer-project/homework/HW6/data/pq/green"
)

In [6]:
df_green.createOrReplaceTempView('green')

In [13]:
df_green_revenue = spark.sql("""
SELECT  
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('hour',lpep_pickup_datetime) AS hour, 

    -- Revenue calculation 
  
    SUM(total_amount) AS amount,
   count(1) AS number_records

FROM
    green
where year(lpep_pickup_datetime) >='2020'
GROUP BY
    1, 2
order by 2,1
""")

In [14]:
df_green_revenue.show()

[Stage 8:===================================================>       (7 + 1) / 8]

+------------+-------------------+------------------+--------------+
|revenue_zone|               hour|            amount|number_records|
+------------+-------------------+------------------+--------------+
|           7|2020-01-01 00:00:00| 769.7299999999997|            45|
|          17|2020-01-01 00:00:00|195.03000000000003|             9|
|          18|2020-01-01 00:00:00|               7.8|             1|
|          22|2020-01-01 00:00:00|              15.8|             1|
|          24|2020-01-01 00:00:00|              87.6|             3|
|          25|2020-01-01 00:00:00| 531.0000000000001|            26|
|          29|2020-01-01 00:00:00|              61.3|             1|
|          32|2020-01-01 00:00:00| 68.94999999999999|             2|
|          33|2020-01-01 00:00:00|            317.27|            11|
|          35|2020-01-01 00:00:00|            129.96|             5|
|          36|2020-01-01 00:00:00|295.34000000000003|            11|
|          37|2020-01-01 00:00:00|

In [15]:
df_green_revenue.write.parquet('data/report/revenue/green')

26/03/05 19:46:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [16]:
df_yellow = spark.read.option("recursiveFileLookup","true").parquet(
"/Users/ethandu/data-engineer-project/homework/HW6/data/pq/yellow"
)

In [19]:
df_yellow.createOrReplaceTempView('yellow')

In [21]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge']

In [22]:
df_yellow_revenue = spark.sql("""
SELECT  
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('hour',tpep_pickup_datetime) AS hour, 

    -- Revenue calculation 
  
    SUM(total_amount) AS amount,
   count(1) AS number_records

FROM
    yellow
where year(tpep_pickup_datetime) >='2020'
GROUP BY
    1, 2
order by 2,1
""")

In [25]:
df_yellow_revenue \
    .repartition(20)\
    .write\
    .parquet('data/report/revenue/yellow',mode='overwrite')

26/03/05 19:59:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [34]:
df_green_revenue_temp= df_green_revenue\
    .withColumnRenamed('amount','green_cmount')\
    .withColumnRenamed('number_records','green_number_records')
df_yellow_revenue_temp =df_yellow_revenue\
    .withColumnRenamed('amount','yellow_cmount')\
    .withColumnRenamed('number_records','yellow_number_records')

In [35]:
df_join = df_green_revenue_temp.join(df_yellow_revenue_temp,on=['hour','revenue_zone'],how ='outer')

In [36]:
df_join.show()

[Stage 51:======================================>                  (8 + 4) / 12]

+-------------------+------------+------------+--------------------+------------------+---------------------+
|               hour|revenue_zone|green_cmount|green_number_records|     yellow_cmount|yellow_number_records|
+-------------------+------------+------------+--------------------+------------------+---------------------+
|2020-01-01 03:00:00|           1|       155.3|                   1|              95.3|                    1|
|2020-01-01 04:00:00|           1|        NULL|                NULL|              94.8|                    1|
|2020-01-01 07:00:00|           1|        NULL|                NULL|              0.31|                    1|
|2020-01-01 10:00:00|           1|        NULL|                NULL|             100.6|                    2|
|2020-01-01 20:00:00|           1|        NULL|                NULL|             96.96|                    3|
|2020-01-02 01:00:00|           1|        NULL|                NULL|             84.36|                    1|
|2020-01-0

In [38]:
df_join.repartition(20).write.parquet('data/report/revenue/total')

26/03/05 20:13:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [39]:
df_join = spark.read.parquet('data/report/revenue/total')

In [40]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-05 20:25:01--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:234c:ec00:b:20a5:b140:21, 2600:9000:234c:5c00:b:20a5:b140:21, 2600:9000:234c:1000:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:234c:ec00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-05 20:25:02 (27.7 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [41]:
!head taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"


In [42]:
df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [43]:
df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [44]:
df.write.parquet('zones')

In [45]:
df_zones = spark.read.parquet('zones')

In [46]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [50]:
df_result = df_join.join(df_zones,df_join.revenue_zone == df_zones.LocationID)

In [51]:
df_result.show()

+-------------------+------------+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|revenue_zone|      green_cmount|green_number_records|     yellow_cmount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+------------+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-09-06 13:00:00|         100|              NULL|                NULL| 329.9000000000001|                   17|       100|Manhattan|    Garment District| Yellow Zone|
|2021-05-07 13:00:00|         177|            129.45|                   2|              NULL|                 NULL|       177| Brooklyn|          Ocean Hill|   Boro Zone|
|2020-01-05 01:00:00|         126|              9.49|                   1|              NULL|                 NULL|       126|    Bronx|         

In [53]:
df_result.drop('LocationID','revenue_zone').show()

+-------------------+------------------+--------------------+------------------+---------------------+---------+--------------------+------------+
|               hour|      green_cmount|green_number_records|     yellow_cmount|yellow_number_records|  Borough|                Zone|service_zone|
+-------------------+------------------+--------------------+------------------+---------------------+---------+--------------------+------------+
|2020-09-06 13:00:00|              NULL|                NULL| 329.9000000000001|                   17|Manhattan|    Garment District| Yellow Zone|
|2021-05-07 13:00:00|            129.45|                   2|              NULL|                 NULL| Brooklyn|          Ocean Hill|   Boro Zone|
|2020-01-05 01:00:00|              9.49|                   1|              NULL|                 NULL|    Bronx|         Hunts Point|   Boro Zone|
|2020-02-17 01:00:00|            126.86|                  10|197.26000000000005|                    9|Manhattan|      

In [54]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')